# Study 949 — Riding the TIPS Curve — the teardown

The excess-of-cash ladder race, the duration-hedged residual with a one-day-lagged rolling beta, Newey-West *t*s, block-bootstrap CIs, the 2021-2023 era cut (gross **and** net), a one-year jackknife, cost/borrow/pairing/window sweeps, a multiplicity audit over every *t* here, and the live synthetic control.

Every real number is frozen from `docs/results.md` (Fingerprint `9da861d9d566`), window 2012-10-16 → 2026-06-30, 3444 daily observations, as-of 2026-06-30. Daily **total-return** closes; the cash leg is BIL's realised total return (1.59%/yr over the window).

In [1]:
R = {'start': '2012-10-16', 'end': '2026-06-30', 'n_days': 3444, 'fp': '9da861d9d566', 'cash_cagr': 1.59, 'ladder': {'VTIP': {'dur': '~2.5y', 'exret': 0.65, 'vol': 2.54, 'sharpe': 0.256, 't': 0.99, 'exdd': -7.4, 'cagr': 2.23, 'absdd': -6.3, 'ci': (-0.193, 0.767), 'neg': 13.6}, 'SCHP': {'dur': '~7y', 'exret': 0.35, 'vol': 5.48, 'sharpe': 0.064, 't': 0.24, 'exdd': -18.4, 'cagr': 1.8, 'absdd': -14.3, 'ci': (-0.409, 0.575), 'neg': 38.8}, 'TIP': {'dur': '~7y', 'exret': 0.27, 'vol': 5.61, 'sharpe': 0.049, 't': 0.19, 'exdd': -18.7, 'cagr': 1.71, 'absdd': -14.5, 'ci': (-0.417, 0.545), 'neg': 41.8}, 'LTPZ': {'dur': '~20y', 'exret': -0.02, 'vol': 14.68, 'sharpe': -0.001, 't': -0.0, 'exdd': -44.8, 'cagr': 0.49, 'absdd': -41.0, 'ci': (-0.462, 0.454), 'neg': 49.8}}, 'diffs': {'SCHP': (-0.3, -0.32, -0.192), 'TIP': (-0.38, -0.39, -0.207), 'LTPZ': (-0.67, -0.21, -0.257)}, 'sleeves': {'VTIP-SHY': {'beta': 1.12, 'gross': 0.98, 'tg': 1.55, 'net': 0.6, 'tn': 0.94, 'sharpe': 0.277, 'vol': 2.16, 'turn': 2.3}, 'SCHP-IEF': {'beta': 0.68, 'gross': 0.69, 'tg': 0.68, 'net': 0.47, 'tn': 0.47, 'sharpe': 0.141, 'vol': 3.35, 'turn': 0.6}, 'TIP-IEF': {'beta': 0.69, 'gross': 0.55, 'tg': 0.54, 'net': 0.33, 'tn': 0.33, 'sharpe': 0.096, 'vol': 3.49, 'turn': 0.6}, 'LTPZ-TLT': {'beta': 0.86, 'gross': 0.34, 'tg': 0.15, 'net': 0.07, 'tn': 0.03, 'sharpe': 0.009, 'vol': 8.22, 'turn': 0.6}}, 'static': {'VTIP~SHY': (0.86, 1.016, 1.47, 0.317), 'SCHP~IEF': (0.46, 0.682, 0.48, 0.637), 'TIP~IEF': (0.38, 0.693, 0.39, 0.627), 'LTPZ~TLT': (0.24, 0.85, 0.11, 0.699)}, 'boot_gross': (0.98, -0.27, 2.11, 5.8), 'boot_net': (0.6, -0.66, 1.73, 15.9), 'eras': {'VTIP-SHY': ((-0.2, -0.22), (2.36, 1.72), (0.79, 1.05)), 'SCHP-IEF': ((0.09, 0.06), (1.54, 0.65), (0.29, 0.26)), 'TIP-IEF': ((-0.05, -0.04), (1.39, 0.59), (0.19, 0.17)), 'LTPZ-TLT': ((0.15, 0.05), (0.18, 0.03), (-0.28, -0.12))}, 'eras_gross': {'VTIP-SHY': ((0.18, 0.19), (2.81, 2.04), (1.09, 1.45)), 'SCHP-IEF': ((0.31, 0.22), (1.75, 0.74), (0.5, 0.44)), 'TIP-IEF': ((0.17, 0.12), (1.6, 0.68), (0.39, 0.35)), 'LTPZ-TLT': ((0.42, 0.13), (0.44, 0.08), (-0.02, -0.01))}, 'era_spans': (('pre-shock', '2013-10-21', '2020-12-31', 1813, 7.2), ('shock', '2021-01-04', '2023-12-29', 753, 3.0), ('post-shock', '2024-01-02', '2026-06-30', 625, 2.5)), 'tmax': [(2.04, "VTIP-SHY gross, sub-window 'shock 2021-2023'"), (1.72, "VTIP-SHY net, sub-window 'shock 2021-2023'"), (1.55, 'VTIP-SHY sleeve, gross, 252d beta  [best FULL-SAMPLE]'), (1.47, 'VTIP~SHY full-sample OLS alpha (IN-SAMPLE hedge)'), (1.45, "VTIP-SHY gross, sub-window 'post-shock 2024+'"), (1.24, 'VTIP-SHY sleeve, net, 504d beta'), (1.24, 'VTIP-IEI sleeve, gross, 252d beta'), (1.09, 'VTIP-IEF sleeve, gross, 252d beta')], 'n_tests': 56, 'n_over_2': 1, 'bonferroni': 3.2, 't_max_full_sample': 1.55, 'ex2021': {'VTIP-SHY': (0.53, 0.8, 0.17, 0.26), 'SCHP-IEF': (0.02, 0.02, -0.19, -0.19), 'TIP-IEF': (-0.1, -0.09, -0.32, -0.3), 'LTPZ-TLT': (-0.59, -0.25, -0.86, -0.37)}, 'years': {2013: -0.47, 2014: -2.36, 2015: -0.54, 2016: 1.54, 2017: 0.55, 2018: -0.86, 2019: 1.53, 2020: 1.79, 2021: 6.38, 2022: 1.63, 2023: 0.51, 2024: 0.63, 2025: 1.19, 2026: 0.87}, 'cost_sweep': [(0.0, 0.64, 1.02), (2.0, 0.6, 0.94), (5.0, 0.53, 0.84), (10.0, 0.42, 0.66)], 'borrow_sweep': [(0.0, 0.93, 1.47, 0.433), (25.0, 0.65, 1.03, 0.303), (30.0, 0.6, 0.94, 0.277), (50.0, 0.37, 0.59, 0.173), (100.0, -0.19, -0.29, -0.087)], 'pairings': [('VTIP', 'SHY', 1.12, 0.98, 1.55, 0.6, 0.94), ('VTIP', 'IEI', 0.37, 0.76, 1.24, 0.64, 1.04), ('VTIP', 'IEF', 0.19, 0.68, 1.09, 0.61, 0.99), ('SCHP', 'IEF', 0.68, 0.69, 0.68, 0.47, 0.47), ('SCHP', 'IEI', 1.17, 0.86, 0.81, 0.49, 0.46), ('LTPZ', 'TLT', 0.86, 0.34, 0.15, 0.07, 0.03), ('LTPZ', 'IEF', 1.83, 0.64, 0.26, 0.06, 0.02)], 'windows': [(126, 0.28, 0.45), (252, 0.6, 0.94), (504, 0.84, 1.24)], 'syn_planted': (2.0, 2.19, 3.65), 'syn_null': (0.19, 0.32), 'syn_seeds': (0.5, 1.04, 0, 8), 'syn_panel': (2.07, 2.08, 1.55, 2.0)}

## 1. Design, and the one execution lag

Two engines share one convention.

- **Ladder race.** Buy-and-hold funds, excess of BIL's actual total return. No signal, no turnover, no costs to charge — the expense ratio is already inside the NAV.
- **Duration-hedged residual.** `resid_t = ex_tips_t − beta_{t−1} · ex_nom_t`, with `beta_{t−1}` from a trailing 252-day OLS ending at *t*−1. **That is the study's only execution lag**, and it is the only place anything could peek: the legs themselves have no timing rule.

**Frictions.** One-way cost on |Δbeta| × NAV (the long leg is buy-and-hold and pays nothing); borrow on |beta| × NAV accrued daily. Base case 2 bps and 30 bps/yr — **the borrow rate is a PROXY** and is swept below.

**Identification, stated once and honoured throughout.** A long-linker / short-nominal residual is a *long-breakeven* position: E[resid] = roll-down in real yields **+** (realised inflation − breakeven at entry). Fund total returns cannot separate them. A positive residual is therefore evidence of *something* the duration hedge misses — not proof of roll-down.

> 💡 **In plain words.** We can measure what a linker earns beyond an ordinary bond of the same maturity, but we cannot tell whether that extra came from sliding down the real curve or from inflation simply arriving higher than expected. So we check *when* it arrived.

## 2. The ladder, excess-of-cash

In [2]:
print(f"{'leg':6s}{'dur':>7s}{'exret':>9s}{'vol':>8s}{'exSh':>8s}"
      f"{'HAC t':>8s}{'exDD':>9s}{'absCAGR':>10s}{'absDD':>9s}{'Sharpe 95% CI':>22s}")
for k in ('VTIP','SCHP','TIP','LTPZ'):
    d = R['ladder'][k]
    ci = f"[{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]"
    print(f"{k:6s}{d['dur']:>7s}{d['exret']:+8.2f}%{d['vol']:7.2f}%"
          f"{d['sharpe']:+8.3f}{d['t']:+8.2f}{d['exdd']:+8.1f}%"
          f"{d['cagr']:+9.2f}%{d['absdd']:+8.1f}%{ci:>22s}")
print()
for k, (dif, t, gap) in R['diffs'].items():
    print(f"  {k:5s} - VTIP: {dif:+.2f}%/yr  HAC t {t:+.2f}  Sharpe gap {gap:+.3f}")

leg       dur    exret     vol    exSh   HAC t     exDD   absCAGR    absDD         Sharpe 95% CI
VTIP    ~2.5y   +0.65%   2.54%  +0.256   +0.99    -7.4%    +2.23%    -6.3%      [-0.193, +0.767]
SCHP      ~7y   +0.35%   5.48%  +0.064   +0.24   -18.4%    +1.80%   -14.3%      [-0.409, +0.575]
TIP       ~7y   +0.27%   5.61%  +0.049   +0.19   -18.7%    +1.71%   -14.5%      [-0.417, +0.545]
LTPZ     ~20y   -0.02%  14.68%  -0.001   -0.00   -44.8%    +0.49%   -41.0%      [-0.462, +0.454]

  SCHP  - VTIP: -0.30%/yr  HAC t -0.32  Sharpe gap -0.192
  TIP   - VTIP: -0.38%/yr  HAC t -0.39  Sharpe gap -0.207
  LTPZ  - VTIP: -0.67%/yr  HAC t -0.21  Sharpe gap -0.257


Monotone the wrong way, and uniformly insignificant. Long TIPS returned **+0.49%/yr** absolute against the cash leg's **+1.59%/yr**, for a **-41%** drawdown. Real duration was uncompensated on this window; the bootstrap CIs straddle zero for every bucket, so the defensible claim is *no reward*, not *negative reward*.

## 3. Duration-hedged residual carry

Rolling 252-day beta, lagged one day. Net = gross − 2 bps one-way on |Δbeta| − 30 bps/yr borrow on |beta|.

In [3]:
print(f"{'sleeve':11s}{'beta':>6s}{'gross':>9s}{'t':>7s}{'net':>9s}"
      f"{'t':>7s}{'netSh':>8s}{'vol':>8s}{'turnover':>10s}")
for k in ('VTIP-SHY','SCHP-IEF','TIP-IEF','LTPZ-TLT'):
    d = R['sleeves'][k]
    print(f"{k:11s}{d['beta']:6.2f}{d['gross']:+8.2f}%{d['tg']:+7.2f}"
          f"{d['net']:+8.2f}%{d['tn']:+7.2f}{d['sharpe']:+8.3f}"
          f"{d['vol']:7.2f}%{d['turn']:9.1f}x")
g, glo, ghi, gneg = R['boot_gross']; n, nlo, nhi, nneg = R['boot_net']
print(f"\nVTIP-SHY block bootstrap (2,000 draws, 21-day blocks):")
print(f"  gross {g:+.2f}%/yr  95% CI [{glo:+.2f}%, {ghi:+.2f}%]  share<0 {gneg:.1f}%")
print(f"  net   {n:+.2f}%/yr  95% CI [{nlo:+.2f}%, {nhi:+.2f}%]  share<0 {nneg:.1f}%")
print('\nfull-sample OLS (IN-SAMPLE hedge, reference only):')
for k, (a, b, t, r2) in R['static'].items():
    print(f"  {k:10s} alpha {a:+.2f}%/yr  beta {b:.3f}  HAC t {t:+.2f}  R2 {r2:.3f}")

sleeve       beta    gross      t      net      t   netSh     vol  turnover
VTIP-SHY     1.12   +0.98%  +1.55   +0.60%  +0.94  +0.277   2.16%      2.3x
SCHP-IEF     0.68   +0.69%  +0.68   +0.47%  +0.47  +0.141   3.35%      0.6x
TIP-IEF      0.69   +0.55%  +0.54   +0.33%  +0.33  +0.096   3.49%      0.6x
LTPZ-TLT     0.86   +0.34%  +0.15   +0.07%  +0.03  +0.009   8.22%      0.6x

VTIP-SHY block bootstrap (2,000 draws, 21-day blocks):
  gross +0.98%/yr  95% CI [-0.27%, +2.11%]  share<0 5.8%
  net   +0.60%/yr  95% CI [-0.66%, +1.73%]  share<0 15.9%

full-sample OLS (IN-SAMPLE hedge, reference only):
  VTIP~SHY   alpha +0.86%/yr  beta 1.016  HAC t +1.47  R2 0.317
  SCHP~IEF   alpha +0.46%/yr  beta 0.682  HAC t +0.48  R2 0.637
  TIP~IEF    alpha +0.38%/yr  beta 0.693  HAC t +0.39  R2 0.627
  LTPZ~TLT   alpha +0.24%/yr  beta 0.850  HAC t +0.11  R2 0.699


Positive in all four buckets, significant in none; the maximum |*t*| on the **full sample** is **1.55**. The term structure of the residual **declines with duration**, which is the wrong shape for a slope-driven roll-down and the right shape for a short-dated inflation-accrual effect. The in-sample OLS alphas reproduce both the level and the ordering, so this is not an artefact of the rolling estimator.

> 💡 **In plain words.** If you were being paid for sliding down a sloped curve, the payment should grow the further out you sit. It does the opposite.

## 4. Era cut — the 2021-2023 inflation shock, **gross and net**

The `pre-shock` era begins at the sleeve's **first day, 2013-10-21** — the 252-day beta warmup eats the first year — so it is **1,813 days ≈ 7.2 years**, not eight. Gross is shown alongside net because that is where the study's only |*t*| ≥ 2 sits, and a net-only table would have hidden it.

In [4]:
for tag, s, e, nd, yrs in R['era_spans']:
    print(f"  {tag:11s} {s} -> {e}  ({nd:4d}d, {yrs:.1f}y)")
print()
print(f"{'sleeve':11s}{'':>4s}{'pre-shock':>18s}{'shock 21-23':>18s}{'post 24+':>18s}")
for k in R['eras']:
    for tag, src in (('gross', R['eras_gross']), ('net', R['eras'])):
        (a, ta), (b, tb), (c, tc) = src[k]
        hit = '   <== only |t|>=2 in the study' if abs(tb) >= 2 else ''
        print(f"{k:11s}{tag:>6s}{a:+10.2f}% (t{ta:+5.2f}){b:+9.2f}% "
              f"(t{tb:+5.2f}){c:+9.2f}% (t{tc:+5.2f}){hit}")
print('\njackknife -- drop 2021 entirely (RETURNS excised; the hedge beta is')
print('NOT re-estimated without 2021, so early-2022 betas still see it):')
for k, (g, tg, n, tn) in R['ex2021'].items():
    print(f"  {k:11s} gross {g:+.2f}%/yr (t {tg:+.2f})   net {n:+.2f}%/yr (t {tn:+.2f})")

  pre-shock   2013-10-21 -> 2020-12-31  (1813d, 7.2y)
  shock       2021-01-04 -> 2023-12-29  ( 753d, 3.0y)
  post-shock  2024-01-02 -> 2026-06-30  ( 625d, 2.5y)

sleeve                  pre-shock       shock 21-23          post 24+
VTIP-SHY    gross     +0.18% (t+0.19)    +2.81% (t+2.04)    +1.09% (t+1.45)   <== only |t|>=2 in the study
VTIP-SHY      net     -0.20% (t-0.22)    +2.36% (t+1.72)    +0.79% (t+1.05)
SCHP-IEF    gross     +0.31% (t+0.22)    +1.75% (t+0.74)    +0.50% (t+0.44)
SCHP-IEF      net     +0.09% (t+0.06)    +1.54% (t+0.65)    +0.29% (t+0.26)
TIP-IEF     gross     +0.17% (t+0.12)    +1.60% (t+0.68)    +0.39% (t+0.35)
TIP-IEF       net     -0.05% (t-0.04)    +1.39% (t+0.59)    +0.19% (t+0.17)
LTPZ-TLT    gross     +0.42% (t+0.13)    +0.44% (t+0.08)    -0.02% (t-0.01)
LTPZ-TLT      net     +0.15% (t+0.05)    +0.18% (t+0.03)    -0.28% (t-0.12)

jackknife -- drop 2021 entirely (RETURNS excised; the hedge beta is
NOT re-estimated without 2021, so early-2022 betas still se

The **7.2** pre-shock years produced **+0.18%/yr gross (*t* = +0.19)** and **-0.20%/yr net** on the headline sleeve — nothing, with the net sign against the claim. Everything the study found is in and after the shock, and removing **2021 alone** (+6.38% gross that year, against a +0.98%/yr full-sample mean) leaves **+0.17%/yr, *t* = +0.26**, and turns three of the four sleeves negative. This is the study's decisive result: the residual behaves exactly like a long-breakeven leg meeting an inflation surprise, and not at all like a structural roll-down.

**And the *t* = +2.04 in the shock column does not rescue it — it indicts it.** A roll-down carry is a property of a sloped curve; it cannot switch on for exactly the three years CPI surprised and stay dark for the other ten. §8 puts that number in its multiplicity context.

## 5. Calendar years of the headline sleeve (VTIP − SHY, gross %)

**2013 (n = 50, beta warmup) and 2026 (n = 123, as-of cut) are part years**, so the unweighted mean below is a tally of rows, not a return.

In [5]:
for y, v in R['years'].items():
    flag = '  <- 4x the sample mean' if y == 2021 else ''
    if y in (2013, 2026):
        flag = '  (PART year -- not a full 12 months)'
    print(f"  {y}: {v:+6.2f}%{flag}")
vals = [v for y, v in R['years'].items() if y < 2021]
print(f"\npre-2021 rows: {len(vals)} (one of them a 50-day stub), of which "
      f"negative: {sum(1 for v in vals if v < 0)}; "
      f"unweighted mean {sum(vals)/len(vals):+.2f}%")

  2013:  -0.47%  (PART year -- not a full 12 months)
  2014:  -2.36%
  2015:  -0.54%
  2016:  +1.54%
  2017:  +0.55%
  2018:  -0.86%
  2019:  +1.53%
  2020:  +1.79%
  2021:  +6.38%  <- 4x the sample mean
  2022:  +1.63%
  2023:  +0.51%
  2024:  +0.63%
  2025:  +1.19%
  2026:  +0.87%  (PART year -- not a full 12 months)

pre-2021 rows: 8 (one of them a 50-day stub), of which negative: 4; unweighted mean +0.15%


## 6. Friction sweeps (VTIP − SHY)

Borrow is a *level* charge — the sleeve is short every day — so it dominates trading cost. The borrow rate is a **PROXY**, hence the sweep.

In [6]:
print('one-way cost sweep (borrow fixed at 30 bps/yr):')
for c, net, t in R['cost_sweep']:
    print(f"  {c:5.1f} bps -> net {net:+.2f}%/yr  t {t:+.2f}")
print('\nborrow sweep (one-way cost fixed at 2 bps):')
for bo, net, t, sh in R['borrow_sweep']:
    tag = '  <- base case' if bo == 30 else ('  <- edge extinguished' if net < 0 else '')
    print(f"  {bo:5.1f} bps/yr -> net {net:+.2f}%/yr  t {t:+.2f}  Sharpe {sh:+.3f}{tag}")

one-way cost sweep (borrow fixed at 30 bps/yr):
    0.0 bps -> net +0.64%/yr  t +1.02
    2.0 bps -> net +0.60%/yr  t +0.94
    5.0 bps -> net +0.53%/yr  t +0.84
   10.0 bps -> net +0.42%/yr  t +0.66

borrow sweep (one-way cost fixed at 2 bps):
    0.0 bps/yr -> net +0.93%/yr  t +1.47  Sharpe +0.433
   25.0 bps/yr -> net +0.65%/yr  t +1.03  Sharpe +0.303
   30.0 bps/yr -> net +0.60%/yr  t +0.94  Sharpe +0.277  <- base case
   50.0 bps/yr -> net +0.37%/yr  t +0.59  Sharpe +0.173
  100.0 bps/yr -> net -0.19%/yr  t -0.29  Sharpe -0.087  <- edge extinguished


## 7. Pairing and window robustness

The linker↔nominal duration match is an **assumption** taken from sponsor fact sheets, so it gets swept; the hedge ratio itself is always estimated from returns, so the pairing only chooses the factor.

In [7]:
for a, b, be, g, tg, n, tn in R['pairings']:
    print(f"  {a:5s} hedged with {b:4s}: beta {be:5.2f}  gross {g:+.2f}% "
          f"(t {tg:+.2f})  net {n:+.2f}% (t {tn:+.2f})")
print('\nbeta estimation window (VTIP-SHY):')
for w, net, t in R['windows']:
    print(f"  {w:3d} days -> net {net:+.2f}%/yr  t {t:+.2f}")

  VTIP  hedged with SHY : beta  1.12  gross +0.98% (t +1.55)  net +0.60% (t +0.94)
  VTIP  hedged with IEI : beta  0.37  gross +0.76% (t +1.24)  net +0.64% (t +1.04)
  VTIP  hedged with IEF : beta  0.19  gross +0.68% (t +1.09)  net +0.61% (t +0.99)
  SCHP  hedged with IEF : beta  0.68  gross +0.69% (t +0.68)  net +0.47% (t +0.47)
  SCHP  hedged with IEI : beta  1.17  gross +0.86% (t +0.81)  net +0.49% (t +0.46)
  LTPZ  hedged with TLT : beta  0.86  gross +0.34% (t +0.15)  net +0.07% (t +0.03)
  LTPZ  hedged with IEF : beta  1.83  gross +0.64% (t +0.26)  net +0.06% (t +0.02)

beta estimation window (VTIP-SHY):
  126 days -> net +0.28%/yr  t +0.45
  252 days -> net +0.60%/yr  t +0.94
  504 days -> net +0.84%/yr  t +1.24


No pairing rescues the result and no window does either — the most flattering choice (504-day beta) reaches *t* = +1.24 at **+0.84%/yr net**, which is the **largest net figure anywhere in this study** (the headline row is the *base case*, not the best one). There is no specification here under which the residual clears the bar on the full sample.

## 8. The multiplicity audit — every real-tape *t*, ranked

The headline sleeve VTIP−SHY was chosen **after the fact** as the best of four, so the study owes you the whole list rather than its favourite row. It computes **56** HAC *t*s on the real tape. Top of that ranking:

In [8]:
TMAX = [(2.04, "VTIP-SHY gross, sub-window 'shock 2021-2023'"), (1.72, "VTIP-SHY net, sub-window 'shock 2021-2023'"), (1.55, 'VTIP-SHY sleeve, gross, 252d beta  [best FULL-SAMPLE]'), (1.47, 'VTIP~SHY full-sample OLS alpha (IN-SAMPLE hedge)'), (1.45, "VTIP-SHY gross, sub-window 'post-shock 2024+'"), (1.24, 'VTIP-SHY sleeve, net, 504d beta'), (1.24, 'VTIP-IEI sleeve, gross, 252d beta'), (1.09, 'VTIP-IEF sleeve, gross, 252d beta')]
N_TESTS = 56
N_OVER = 1
BONF = 3.2
T_FULL = 1.55
for t, name in TMAX:
    flag = '   <== CLEARS |t|>=2' if abs(t) >= 2.0 else ''
    print(f"  t {t:+5.2f}  {name}{flag}")
print(f"\n  {N_OVER} of {N_TESTS} statistics reach |t| >= 2; "
      f"the Bonferroni bar for {N_TESTS} tests is |t| ~ {BONF:.1f}.")
print(f"  On the FULL sample nothing exceeds |t| = {T_FULL:.2f}.")

  t +2.04  VTIP-SHY gross, sub-window 'shock 2021-2023'   <== CLEARS |t|>=2
  t +1.72  VTIP-SHY net, sub-window 'shock 2021-2023'
  t +1.55  VTIP-SHY sleeve, gross, 252d beta  [best FULL-SAMPLE]
  t +1.47  VTIP~SHY full-sample OLS alpha (IN-SAMPLE hedge)
  t +1.45  VTIP-SHY gross, sub-window 'post-shock 2024+'
  t +1.24  VTIP-SHY sleeve, net, 504d beta
  t +1.24  VTIP-IEI sleeve, gross, 252d beta
  t +1.09  VTIP-IEF sleeve, gross, 252d beta

  1 of 56 statistics reach |t| >= 2; the Bonferroni bar for 56 tests is |t| ~ 3.2.
  On the FULL sample nothing exceeds |t| = 1.55.


**1 of 56** clears the nominal bar, against a Bonferroni threshold of ≈**3.2** — and that one is *gross*, *post-hoc*, and confined to the three-year window the study already names as the rival explanation. Charge the borrow the permanently-short leg actually owes and it falls to +1.72. No adjustment is needed to reach the null here; what is needed is disclosure of how many numbers were looked at, which is what this table is.

## 9. Synthetic control — is the detector powered and unbiased?

*Live, offline, synthetic. Nothing below touches the real tape.* A planted 2%/yr residual carry must be recovered with a decisive *t*; the null must stay quiet across seeds; and a panel with the **same** carry in every bucket must come back with a **flat** carry term structure — otherwise the real tape's declining term structure would be an estimator artefact.

In [9]:
import os, sys
import numpy as np
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from tips_roll import data, strategy as st
pl = st.synthetic_detect(data.synthetic_daily(signal_strength=1.0, seed=949)[0])
nu = st.synthetic_detect(data.synthetic_daily(signal_strength=0.0, seed=949)[0])
print('planted 2.00%%/yr carry : recovered %+.2f%%/yr  HAC t %+.2f  (beta %.2f vs 0.85 planted)'
      % (pl['gross_ann']*100, pl['t_gross'], pl['mean_beta']))
print('null, no carry planted : recovered %+.2f%%/yr  HAC t %+.2f'
      % (nu['gross_ann']*100, nu['t_gross']))
ts = np.array([st.synthetic_detect(
        data.synthetic_daily(signal_strength=0.0, seed=949+s)[0])['t_gross']
    for s in range(8)])
print('null across 8 seeds    : mean t %+.2f  sd %.2f  fires |t|>=2 on %d/8'
      % (ts.mean(), ts.std(ddof=1), int((np.abs(ts) >= 2).sum())))
panel, truth = data.synthetic_panel(signal_strength=1.0, seed=949)
dl = st.synthetic_ladder_detect(panel, truth)
rec = [dl['carries']['bucket_%d' % i]['gross_ann']*100 for i in range(truth['n_buckets'])]
print('ladder panel, same carry in all 3 buckets: recovered '
      + ', '.join('%+.2f%%' % c for c in rec)
      + '  (planted %+.2f%%) -> flat, as it should be' % (dl['planted_carry_ann']*100,))

planted 2.00%/yr carry : recovered +2.19%/yr  HAC t +3.65  (beta 0.83 vs 0.85 planted)
null, no carry planted : recovered +0.19%/yr  HAC t +0.32


null across 8 seeds    : mean t +0.50  sd 1.04  fires |t|>=2 on 0/8


ladder panel, same carry in all 3 buckets: recovered +2.07%, +2.08%, +1.55%  (planted +2.00%) -> flat, as it should be


Powered (*t* > 3 on a planted 2%/yr carry, on a **shorter** sample than the real one), unbiased (0/8 false positives on the null), and free of a spurious term-structure tilt. Had a real-yield roll-down carry of the advertised size been present in the tape, this machinery would have stamped it.

## Verdict

**Signal — None.** Both forms of the claim fail. Extending along the real curve produced a *monotonically falling* excess Sharpe (+0.256 → -0.001) with every long-minus-short difference insignificant and every Sharpe CI straddling zero. The duration-hedged residual is positive in all four buckets and significant in none (max **full-sample** HAC *t* **+1.55** gross, **+0.94** net; bootstrap CI [-0.66%, +1.73%]), it declines with duration, and it is entirely post-2020 (+0.18%/yr gross / -0.20%/yr net across the 7.2 pre-shock years, +2.36%/yr in 2021-2023, +0.17%/yr ex-2021). Given that the residual is by construction a long-breakeven leg, the parsimonious reading is an inflation surprise, not roll-down — and the two are not separately identifiable here, which is itself a reason not to award a green stamp.

**The one *t* ≥ 2, disclosed.** Exactly 1 of the 56 statistics computed on the real tape reaches 2: **+2.04**, the gross carry of the best-of-four sleeve inside the hand-picked shock window (§8). Against a Bonferroni bar of ≈3.2, and landing exactly where the rival hypothesis says it should, it does not move the stamp.

**Tradability — Mirage.** Best net figure anywhere +0.84%/yr (504-day beta, *t* = +1.24); base case +0.60%/yr at 2.16% vol, from a permanently short spread that turns negative at ~100 bps/yr borrow and survives no single-year jackknife. The long-only version — just buying LTPZ — under-earned T-bills by 1.10 pp/yr while drawing down -41%.

**Sample caveat, named on the Signal axis.** VTIP's 2012-10 inception confines the whole study to the post-GFC real-rate regime — no 2008-2009 TIPS liquidity dislocation, no pre-2004 illiquidity premium. What is measured here is a 13.5-year window that happens to contain exactly one inflation shock.